In [2]:
import os
from typing import Any

from dotenv import load_dotenv
from utils.agent_visualizer import print_activity

from claude_code_sdk import ClaudeCodeOptions, ClaudeSDKClient

# 02 - The Observability Agent

In the anterior notebooks we have built a basic research agent e a Chief of Staff multi-agent framework. While the agents we have built are already powerful, they were still limited in o que they could do: the web buscar agent is limited para busca the internet e our Chief of Staff agent was limited para interacting com its own filesystem.

este is a serious constraint: real-world agents frequentemente need para interact com other systems like databases, APIs, arquivo systems, e other specialized services. [MCP (Model Context Protocol)](https://modelcontextprotocol.io/docs/getting-started/intro) is an abrir-source standard for AI-tool integrations aquele allows for an fácil connection entre our agents e estes externo systems. In este notebook, we will explore como para conectar MCP servers para our agent.

**Need mais details on MCP?** For comprehensive Configuração instructions, Configuração best practices, e troubleshooting tips, Veja o [Claude Code MCP Documentação](https://docs.anthropic.com/en/docs/claude-code/mcp).

## Introdução para the MCP servidor
### 1. The Git MCP servidor

Let's primeiro give our agent the ability para understand e work com Git repositories. By adding the [Git MCP servidor](https://github.com/modelcontextprotocol/servers/tree/principal/src/git) para our agent, it gains access para 13 Git-specific tools aquele let it examine commit history, verificar arquivo changes, criar branches, e even make commits. este transforms our agent de a passivo observer into an ativo participant in your desenvolvimento fluxo de trabalho. In Este exemplo, we'll configurar the agent para explore a repositório's history using only Git tools. este is pretty simples, but knowing este, it is não difficult para imagine agents aquele can automatically criar pull requests, analyze code evolution patterns, ou help manage complexo Git workflows across multiple repositories.

In [ ]:
# define our git MCP server (it was downloaded when you ran uv sync as it is defined in the pyproject.toml file)
git_mcp: dict[str, Any] = {
    "git": {
        "command": "uv",
        "args": ["run", "python", "-m", "mcp_server_git", "--repository", os.getcwd()],
    }
}

In [4]:
messages = []
async with (
    ClaudeSDKClient(
        options=ClaudeCodeOptions(
            model="claude-sonnet-4-20250514",
            mcp_servers=git_mcp,
            allowed_tools=[
                "mcp__git"
            ],  # For MCP tools, in allowed tools we must add the mcp__serverName__toolName format or mcp__serverName to enable all
            permission_mode="acceptEdits",  # auto-accept file edit permissions
        )
    ) as agent
):
    await agent.query(
        "Use ONLY your git mcp tools to quickly explore this repo's history and gimme a brief summary."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Thinking...
🤖 Using: mcp__git()
✓ Tool completed
🤖 Thinking...
🤖 Using: Bash()
🤖 Using: Bash()
🤖 Using: Bash()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


In [ ]:
print(f"\nResult:\n{messages[-1].result}")

### 2. The GitHub MCP servidor

agora let's level up de local Git operations para full GitHub platform integração. By switching para the [official GitHub MCP servidor](https://github.com/github/github-mcp-servidor/tree/principal), our agent gains access para over 100 tools aquele interact com GitHub's entire ecosystem – de managing issues e pull requests para monitoramento CI/CD workflows e analyzing code segurança alerts. este servidor can work com both público e privado repositories, giving your agent the ability para automate complexo GitHub workflows aquele would typically require multiple manual steps.

#### Step 1: Set up your GitHub Token

You need a GitHub Personal Access Token. Get one [aqui](https://github.com/configurações/personal-access-tokens/novo) e put in the .env arquivo as ```GITHUB_TOKEN="<token>"```
> Note: When getting your token, select "Fine-grained" token with the default options (i.e., public repos, no account permissions), that'll be the easiest way to get this demo working.

Also, for Este exemplo you will have para have [Docker](https://www.docker.com/products/docker-desktop/) running on your machine. Docker is obrigatório because the GitHub MCP servidor runs in a containerized environment for segurança e isolation.

**Docker Quick Configuração:**
- instalar Docker Desktop de [docker.com](https://www.docker.com/products/docker-desktop/)
- Ensure Docker is running (you'll Veja o Docker ícone in your system tray)
- verificar com `docker --version` in your terminal
- **Troubleshooting:** se Docker won't iniciar, verificar aquele virtualization is habilitado in your BIOS. For detailed Configuração instructions, Veja o [Docker Documentação](https://docs.docker.com/get-docker/)

#### Step 2: Define the mcp servidor e iniciar the agent loop!

In [6]:
# define our github mcp server
load_dotenv(override=True)
github_mcp: dict[str, Any] = {
    "github": {
        "command": "docker",
        "args": [
            "run",
            "-i",
            "--rm",
            "-e",
            "GITHUB_PERSONAL_ACCESS_TOKEN",
            "ghcr.io/github/github-mcp-server",
        ],
        "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": os.environ.get("GITHUB_TOKEN")},
    }
}

In [7]:
# run our agent
messages = []
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        mcp_servers=github_mcp,
        allowed_tools=["mcp__github"],
        permission_mode="acceptEdits",  # auto-accept permissions
    )
) as agent:
    await agent.query(
        "Use ONLY your GitHub MCP tools to search for the anthropics/claude-code-sdk-python repository and and give me a couple facts about it"
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Thinking...
🤖 Using: mcp__github__search_repositories()
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_file_contents()
✓ Tool completed
🤖 Thinking...


In [ ]:
print(f"\nResult:\n{messages[-1].result}")

## Real use case: An observability agent

agora, com such simples Configuração we can already have an agent acting as self-healing software system!

In [9]:
load_dotenv(override=True)

prompt = """Monitor the GitHub Actions workflows for facebook/react.
Look at the last triggered CI pipeline. 
1. Analyze the trigger for the pipeline
2. Identify whether the pipeline passed or not
3. If it failed, explain which test failed
4. Identify whether human involvement is required

IMPORTANT: Do not raise a PR, issue, or bug on github yet. Just give me a summary of your findings and plan.

Focus on the 'CI' workflow specifically. Use your Github MCP server tools!"""

github_mcp: dict[str, Any] = {
    "github": {
        "command": "docker",
        "args": [
            "run",
            "-i",
            "--rm",
            "-e",
            "GITHUB_PERSONAL_ACCESS_TOKEN",
            "ghcr.io/github/github-mcp-server",
        ],
        "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": os.environ.get("GITHUB_TOKEN")},
    }
}

messages = []
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        mcp_servers=github_mcp,
        allowed_tools=["mcp__github"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(prompt)
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Using: mcp__github__list_workflows()
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__list_workflow_runs()
✓ Tool completed
🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_workflow_run()
✓ Tool completed
🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Using: mcp__github__get_job_logs()
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__list_workflow_jobs()
✓ Tool completed
🤖 Thinking...
🤖 Using: WebFetch()
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_job_logs()
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_job_logs()
✓ Tool completed
🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Using: mcp__github__get_job_logs()
✓ Tool completed
🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Thinking...


In [ ]:
print(f"\nResult:\n{messages[-1].result}")

### Observability Agent as módulo

The `observability_agent/agent.py` arquivo contains the same minimal helper funções as the research agent ou chief of staff agent, just enhanced for GitHub monitoramento. 

As antes, para use it as a módulo in your Python code:

In [ ]:
from observability_agent.agent import send_query

result = await send_query(
    "Check the CI status for the last 2 runs in anthropics/claude-code-sdk-python. Just do 3 tool calls, be efficient."
)
print(f"Monitoring result: {result}")

We can do multi-turn conversations com este agent as well:

In [ ]:
# Example 2: Multi-turn conversation for deeper monitoring
result1 = await send_query("What's the current CI status for facebook/react?")
print(f"Initial check: {result1[:250]}...\n")

In [ ]:
# Continue the conversation to dig deeper
result2 = await send_query(
    "Are there any flaky tests in the recent failures? You can only make one tool call.",
    continue_conversation=True,
)
print(f"Follow-up analysis: {result2[:250]}...")

## Conclusion

We've demonstrated como the Claude Code SDK enables seamless integração com externo systems through the Model Context Protocol (MCP). Starting com local Git operations through the Git MCP servidor, we progressively expanded para full GitHub platform integração com access para over 100 GitHub-specific tools. este transformed our agent de a local assistant into a powerful observability system capable of monitoramento workflows, analyzing CI/CD failures, e providing actionable insights for produção systems.

By connecting MCP servers para our agent, we created an autonomous observability system aquele monitors GitHub Actions workflows, distinguishes entre real failures e segurança restrictions, e provides detailed analysis of teste failures. The system demonstrates como agents can actively participate in your DevOps fluxo de trabalho, moving de passivo monitoramento para intelligent incident resposta.

este concludes, for agora, our journey through the Claude Code SDK Tutorial series. We've progressed de simples research agents para sophisticated multi-agent orchestration, e finally para externo system integração through MCP. Together, estes patterns provide the foundation for building produção-ready agentic systems aquele can handle real-world complexity while maintaining governance, compliance, e observability.

### o que You've Learned Across todos Notebooks

**de Notebook 00 (Research Agent)**
- Core SDK fundamentals com `query()` e `ClaudeSDKClient`
- Basic tool Uso com WebSearch e ler
- simples agent loops e conversation management

**de Notebook 01 (Chief of Staff)**
- avançado Recursos: memória, saída styles, planning mode
- Multi-agent coordination through subagents
- Governance through hooks e personalizado comandos
- Enterprise-ready agent architectures

**de Notebook 02 (Observability Agent)**
- externo system integração via MCP servers
- Real-tempo monitoramento e incident resposta
- produção fluxo de trabalho automação
- escalável agent Implantação patterns

The completo implementations for todos three agents are available in their respective directories (`research_agent/`, `chief_of_staff_agent/`, `observability_agent/`), ready para serve as inspiration for integrations into your produção systems.